In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("FeatureEngineering") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

df = spark.read.parquet("../data/processed/trips_with_disruptions")
print("Rows:", df.count())
df.printSchema()

26/07/14 12:23:46 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
                                                                                

Rows: 765089
root
 |-- operator_name: string (nullable = true)
 |-- line_name: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- trip_date: date (nullable = true)
 |-- weekday_name: string (nullable = true)
 |-- departure_time: string (nullable = true)
 |-- scheduled_datetime: timestamp (nullable = true)
 |-- service_code: string (nullable = true)
 |-- is_disrupted: integer (nullable = true)
 |-- reason: string (nullable = true)
 |-- planned: boolean (nullable = true)
 |-- severity: string (nullable = true)
 |-- overlap_duration_minutes: double (nullable = true)



# extract hour of day from departure_time (stored as "HH:MM:SS" string)

In [2]:
df = df.withColumn("departure_hour", F.split(F.col("departure_time"), ":")[0].cast("int"))

# weekend vs weekday flag, derived from weekday_name we already have
df = df.withColumn(
    "is_weekend",
    F.when(F.col("weekday_name").isin("Saturday", "Sunday"), 1).otherwise(0)
)

# cyclical encoding for hour — so the model understands 23:00 and 00:00 are close,
# not far apart like raw integers would suggest
df = df.withColumn("hour_sin", F.sin(2 * 3.14159265 * F.col("departure_hour") / 24)) \
       .withColumn("hour_cos", F.cos(2 * 3.14159265 * F.col("departure_hour") / 24))

df.select("departure_time", "departure_hour", "weekday_name", "is_weekend", "hour_sin", "hour_cos").show(10)

+--------------+--------------+------------+----------+-------------------+--------------------+
|departure_time|departure_hour|weekday_name|is_weekend|           hour_sin|            hour_cos|
+--------------+--------------+------------+----------+-------------------+--------------------+
|      09:15:00|             9|    Thursday|         0| 0.7071067830903227| -0.7071067792827723|
|      10:45:00|            10|    Thursday|         0| 0.5000000025907099| -0.8660254022886916|
|      16:56:00|            16|    Thursday|         0|-0.8660254013912434| -0.5000000041451357|
|      07:45:00|             7|      Friday|         0| 0.9659258268310472|-0.25881904307982767|
|      07:56:00|             7|      Friday|         0| 0.9659258268310472|-0.25881904307982767|
|      08:56:00|             8|      Friday|         0| 0.8660254049810362|-0.49999999792743216|
|      09:45:00|             9|      Friday|         0| 0.7071067830903227| -0.7071067792827723|
|      09:56:00|             9

# Categorical encoding for ML (StringIndexer + OneHotEncoder via PySpark MLlib):

In [3]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# categorical columns we want the model to learn from
categorical_cols = ["operator_name", "weekday_name", "reason", "severity"]

# StringIndexer converts text categories into numeric indices (required before OneHotEncoder)
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep")
    for col in categorical_cols
]

# OneHotEncoder converts those indices into sparse binary vectors,
# so the model doesn't assume a false ordinal relationship between categories
encoders = [
    OneHotEncoder(inputCol=f"{col}_idx", outputCol=f"{col}_ohe")
    for col in categorical_cols
]

from pyspark.ml import Pipeline

encoding_pipeline = Pipeline(stages=indexers + encoders)
encoding_model = encoding_pipeline.fit(df)
df_encoded = encoding_model.transform(df)

df_encoded.select(
    "operator_name", "operator_name_idx", "operator_name_ohe",
    "severity", "severity_idx", "severity_ohe"
).show(10, truncate=False)

+----------------+-----------------+-----------------+--------+------------+-------------+
|operator_name   |operator_name_idx|operator_name_ohe|severity|severity_idx|severity_ohe |
+----------------+-----------------+-----------------+--------+------------+-------------+
|Arriva Yorkshire|0.0              |(10,[0],[1.0])   |minor   |3.0         |(4,[3],[1.0])|
|Arriva Yorkshire|0.0              |(10,[0],[1.0])   |minor   |3.0         |(4,[3],[1.0])|
|Arriva Yorkshire|0.0              |(10,[0],[1.0])   |minor   |3.0         |(4,[3],[1.0])|
|Arriva Yorkshire|0.0              |(10,[0],[1.0])   |minor   |3.0         |(4,[3],[1.0])|
|Arriva Yorkshire|0.0              |(10,[0],[1.0])   |minor   |3.0         |(4,[3],[1.0])|
|Arriva Yorkshire|0.0              |(10,[0],[1.0])   |minor   |3.0         |(4,[3],[1.0])|
|Arriva Yorkshire|0.0              |(10,[0],[1.0])   |minor   |3.0         |(4,[3],[1.0])|
|Arriva Yorkshire|0.0              |(10,[0],[1.0])   |minor   |3.0         |(4,[3],[1.0])|

**Assembling the final feature vector**

In [4]:
from pyspark.ml.feature import VectorAssembler

# operator_name_ohe removed after diagnostic investigation revealed it reflects
# an artifact of the disruption-to-trip sampling methodology (operators with fewer
# distinct lines were disproportionately swept into "disrupted" whenever even one
# of their lines was randomly sampled) rather than genuine disruption risk —
# see notebook 05 feature importance / operator disruption rate analysis
feature_cols = [
    "hour_sin", "hour_cos", "is_weekend", "weekday_name_ohe"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
df_final = assembler.transform(df_encoded)

df_final.select("features", "is_disrupted").show(5, truncate=False)

+----------------------------------------------------------+------------+
|features                                                  |is_disrupted|
+----------------------------------------------------------+------------+
|(10,[0,1,5],[0.7071067830903227,-0.7071067792827723,1.0]) |1           |
|(10,[0,1,5],[0.5000000025907099,-0.8660254022886916,1.0]) |1           |
|(10,[0,1,5],[-0.8660254013912434,-0.5000000041451357,1.0])|1           |
|(10,[0,1,3],[0.9659258268310472,-0.25881904307982767,1.0])|1           |
|(10,[0,1,3],[0.9659258268310472,-0.25881904307982767,1.0])|1           |
+----------------------------------------------------------+------------+
only showing top 5 rows


# train/test split, done deliberately, not randomly

In [5]:
# time-based split: train on April-May, test on June — avoids leaking
# information about the same disruption events across train/test
split_date = "2026-06-01"

train_df = df_final.filter(F.col("trip_date") < split_date)
test_df = df_final.filter(F.col("trip_date") >= split_date)

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())

print("\nTrain is_disrupted distribution:")
train_df.groupBy("is_disrupted").count().show()

print("Test is_disrupted distribution:")
test_df.groupBy("is_disrupted").count().show()

Train rows: 511133


Test rows: 253956

Train is_disrupted distribution:


+------------+------+
|is_disrupted| count|
+------------+------+
|           1|113117|
|           0|398016|
+------------+------+

Test is_disrupted distribution:
+------------+------+
|is_disrupted| count|
+------------+------+
|           1| 67832|
|           0|186124|
+------------+------+



**Now, Saving the train/test sets so 05_modeling.ipynb can load them independently:**

In [6]:
# save train/test sets for the modeling notebook
train_df.select("features", "is_disrupted").write.mode("overwrite").parquet("../data/processed/train_set")
test_df.select("features", "is_disrupted").write.mode("overwrite").parquet("../data/processed/test_set")

print("Saved train and test sets.")

26/07/14 12:24:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/07/14 12:24:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
[Stage 44:==================================================>       (7 + 1) / 8]

Saved train and test sets.
